In [ ]:
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np
import cv2

from eeg.data_collection import JointData

In [6]:
def hand_detection(mp_hands, mp_drawing, joint_data: list, set_fps: int):
    # getting camera / webcam (0, 1, 2 for connected webcams)
    cap = cv2.VideoCapture(0)

    # set desired FPS to camera
    cap.set(cv2.CAP_PROP_FPS, set_fps)
    # verify FPS
    fps = cap.get(cv2.CAP_PROP_FPS)
    # log FPS for verification
    print(f"Set FPS: {set_fps}\nFPS: {fps}")

    # resource management: open hands as the term below and close it automatically
    # two metrics:
    # 1. detection: threshold for initial detection to be successful
    # 2. tracking: threshold for tracking after initial detection
    with mp_hands.Hands(min_detection_confidence=0.8, min_tracking_confidence=0.5, max_num_hands=1) as hands:
        # reading frames while the capture is opened
        while cap.isOpened():
            # read each frame from webcam
            # ret and frame variables unpacking cap.read() function
            # a return value and image from webcame
            success, frame = cap.read()
            # initialize list for current frame data
            frame_data = []

            if not success:
                print("Ignoring empty camera frame.")
                # if loading a video, use 'break' instead of 'continue'.
                continue

            # to improve performance, optionally mark the image as not writeable to
            # pass by reference.
            frame.flags.writeable = False
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) # model requires 3 channel RGB
            results = hands.process(frame)

            # set flag back to true
            # allows us to draw on image and render
            frame.flags.writeable = True
            frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

            image_height, image_width, _ = frame.shape

            # if solution has landmarks then render results on image
            if results.multi_hand_landmarks:
                # iterate through each landmark
                # hand_landmarks represents the landmarks for that hand
                for hand_landmarks in results.multi_hand_landmarks:
                    # pass in three variables:
                    # 1. image
                    # 2. hand (set of landmarks)
                    # 3. HAND_CONNECTIONS represents the set of coordinates of relations between joints
                    mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)

                    # iterate through hand landmarks list and figure out the position
                    # openCV origin is top left (e.g., y position increases when wrist moves down)
                    for landmark in hand_landmarks.landmark:
                        W = hand_landmarks.landmark[0]
                        relative_x = landmark.x - W.x
                        relative_y = landmark.y - W.y
                        relative_z = landmark.z - W.z
                        # extend list by new landmark positions
                        frame_data.extend([relative_x * image_width, relative_y * image_height, relative_z * image_width])

                # add to data
                joint_data.append(frame_data)

            # render image to screen using OpenCV with "Hand tracking" window title
            cv2.imshow("Hand tracking", frame)

            # hit "q" and close window
            if cv2.waitKey(10) & 0xFF == ord("q"):
                break

    # release webcam
    cap.release()
    # close windows
    cv2.destroyAllWindows()

In [7]:
def write(data: list, filename: str):
    dataset = np.array(data) # turn list into numpy array
    np.save(f"{filename}.npy", dataset) # save numpy array

In [8]:
# helps draw landmarks on screen
mp_drawing = mp.solutions.drawing_utils
# hands model
mp_hands = mp.solutions.hands

# initialize list of frames with landmarks
# each joint is a landmark
joint_data = []
filename = "joint_data"

hand_detection(mp_hands, mp_drawing, joint_data, 30)
write(joint_data, filename)

Set FPS: 30
FPS: 30.0


c:\Users\prabh\Documents\eeg\.venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [17]:
data = JointData("joint_data.npy")
x = data.get_positions("IT")
x

# data.plot_data("PT")

array([[  93.2793808 , -178.30513   ,  -66.50744227],
       [ 105.30527115, -184.66338158,  -60.96606833],
       [ 106.65412903, -185.06698608,  -61.20302546],
       [ 106.56959534, -184.75310326,  -58.26711771],
       [ 106.89487457, -183.96622181,  -59.58449618],
       [ 107.34117508, -183.54535103,  -59.2969091 ],
       [ 106.21242523, -183.30327988,  -59.32958674],
       [ 105.95211029, -183.49378109,  -62.24703705],
       [ 107.98585892, -183.62108231,  -63.87201092],
       [ 108.31552505, -184.55683708,  -62.34432624],
       [ 107.57255554, -183.3497715 ,  -62.70340907],
       [ 107.32849121, -184.9674511 ,  -63.36851798],
       [ 105.72702408, -185.10352135,  -63.2454588 ],
       [ 106.22701645, -185.27961731,  -62.28477802],
       [ 105.58282852, -184.36091423,  -61.8362789 ],
       [ 106.52448654, -184.41307068,  -62.0506876 ],
       [ 107.05831528, -184.85243797,  -61.8288084 ],
       [ 107.80683517, -184.8866272 ,  -61.548218  ],
       [ 106.72088623, -184.

In [ ]:
# arg parse

# review --
# pytorch dataset
# torch.utils.data.Dataset
# data loader

# script to use other hand to start and stop